# CASMI 2026 — tutorial and glossary, from a 0.342 scorer

**Task:** you are given a mass spectrum of an unknown molecule. Name the molecule.
You submit **25 guesses per molecule, in order**, as SMILES strings.

There is a lot of vocabulary here and most of it is never defined anywhere — it is
assumed. This notebook defines all of it in dependency order: the instrument first, then
how a molecule is written down, then how candidates are found and ranked, then the metric,
then de novo generation, then the jargon specific to this competition.

Every computed number comes from the competition's own `train.parquet` / `test.parquet`,
so this runs unchanged on Kaggle or locally. Numbers from my own private holdout are
quoted in tables and labelled as such — never silently mixed in.

In [1]:
import sys, os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

def comp_file(name):
    # locate a competition file on Kaggle or in a local checkout
    for pat in (f"/kaggle/input/**/{name}", f"data/raw/{name}", f"../data/raw/{name}",
                f"../../data/raw/{name}", f"../../../data/raw/{name}"):
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    raise FileNotFoundError(name)

TRAIN, TEST = comp_file("train.parquet"), comp_file("test.parquet")
print("train:", TRAIN)
print("test: ", TEST)

# Kaggle's default image has no RDKit and this kernel runs without internet, so install
# it from the offline wheel dataset attached to the notebook. Locally this is a no-op.
def install_offline_rdkit():
    try:
        import rdkit                    # already available (local checkout)
        return
    except ImportError:
        pass
    import subprocess
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheels = sorted(glob.glob("/kaggle/input/**/rdkit-*.whl", recursive=True))
    ok = [w for w in wheels if f"-{py_tag}-" in os.path.basename(w)]
    if not ok:
        raise RuntimeError(f"no RDKit wheel for {py_tag}; attach an offline rdkit dataset")
    print("installing offline RDKit:", os.path.basename(ok[-1]))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--no-deps", ok[-1]], check=True)

install_offline_rdkit()

from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.Descriptors import ExactMolWt
RDLogger.DisableLog("rdApp.*")

---
# Part 1 — The instrument: what a mass spectrum actually is

A mass spectrometer **weighs molecules**. It cannot see shape, colour or bonds — only
mass. More precisely it measures **mass-to-charge ratio**, **m/z**, because it must charge
the molecule before it can steer it with electric fields.

## Ionisation and the *adduct*

A neutral molecule is invisible to the instrument. To charge it, the machine makes it grab
or lose something — usually a proton. The molecule-plus-whatever-it-grabbed is an
**adduct**.

| adduct | what happened | m/z relative to neutral mass M |
|---|---|---|
| `[M+H]+` | gained a proton | M + 1.00728 |
| `[M-H]-` | lost a proton | M − 1.00728 |
| `[M+Na]+` | grabbed a sodium ion | M + 22.98922 |
| `[M+NH4]+` | grabbed an ammonium ion | M + 18.03383 |

So the **first job** is always to undo this: from the measured m/z and the known adduct,
recover the **neutral mass**. Get the adduct wrong and every later step hunts the wrong
molecule.

In [2]:
PROTON = 1.00727646
adducts = {"[M+H]+": +PROTON, "[M-H]-": -PROTON,
           "[M+Na]+": 22.98922, "[M+NH4]+": 18.03383}
caffeine = Chem.MolFromSmiles("Cn1cnc2c1c(=O)n(C)c(=O)n2C")
M = ExactMolWt(caffeine)
print(f"caffeine neutral (monoisotopic) mass = {M:.5f} Da")
print()
for a, d in adducts.items():
    print(f"  seen as {a:<9} at m/z {M + d:10.5f}")

## MS/MS — why there are two masses

Weighing a molecule gives one number, and thousands of molecules share it. So it is done
twice:

1. **MS1** — weigh everything, then *select* one mass. That ion is the **precursor**, its
   mass the **precursor m/z**.
2. **MS2** — smash the precursor into gas until it breaks, then weigh the pieces. Those
   are **fragment ions** (or *product ions*).

That is **tandem mass spectrometry** — MS/MS, or MS2. The energy used is the **collision
energy**, in electronvolts (**eV**); more energy, smaller pieces.

**A spectrum is therefore a list of (m/z, intensity) pairs** — where the pieces landed and
how many of each. Intensity is normally scaled so the largest peak is 1.0; that peak is the
**base peak**.

The fragments carry the real information: *which* pieces a molecule breaks into depends on
how it is bonded, so the pattern is a fingerprint of the structure.

In [3]:
peaks_mz  = np.array([138.0662, 110.0713, 42.0344, 195.0877, 83.0604, 56.0495])
peaks_int = np.array([1.00,     0.42,     0.31,    0.28,     0.12,    0.08])
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.vlines(peaks_mz, 0, peaks_int, linewidth=2)
ax.plot(peaks_mz, peaks_int, "o", ms=4)
for x, y in zip(peaks_mz, peaks_int):
    ax.annotate(f"{x:.4f}", (x, y), textcoords="offset points", xytext=(0, 6),
                ha="center", fontsize=8)
ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity")
ax.set_title("A (schematic) MS/MS spectrum: caffeine, [M+H]+ precursor 195.0877")
ax.set_ylim(0, 1.18); plt.tight_layout(); plt.show()
print("138.0662 is the BASE PEAK (intensity 1.0 by definition).")
print("195.0877 is surviving precursor. The rest are fragments.")

## ppm — why mass error is quoted in millionths

Instruments are accurate in *relative* terms, so error is quoted in **parts per million**:

    ppm = (m_measured - m_true) / m_true * 1e6

At 400 Da, 5 ppm is 0.002 Da. This matters enormously, because the mass window is how you
decide which structures are worth considering at all. Too wide and you drown in candidates;
too narrow and you exclude the answer.

This competition's test set is entirely **timsTOF**. Here is that instrument's real error
distribution, computed from the training library.

In [4]:
ADD = {"[M+H]+": +PROTON, "[M-H]-": -PROTON, "[M+Na]+": 22.98922,
       "[M+NH4]+": 18.03383, "[M+K]+": 38.96316, "[M+H-H2O]+": -17.00329}
cols = ["normalized_smiles", "precursor_mz", "adduct", "instrument_type"]
t = pq.read_table(TRAIN, columns=cols).to_pandas().dropna()
t = t[t.instrument_type == "timsTOF"].sample(n=25000, random_state=0)
vals = []
for r in t.itertuples():
    d = ADD.get(r.adduct)
    if d is None:
        continue
    m = Chem.MolFromSmiles(r.normalized_smiles)
    if m is None:
        continue
    true = ExactMolWt(m)
    nm = float(r.precursor_mz) - d
    if true > 0 and abs(nm - true) < 0.5:        # same molecule, not a mis-assignment
        vals.append((nm - true) / true * 1e6)
err = np.array(vals)
print(f"timsTOF precursor mass error, n = {len(err):,} real reference spectra")
for q in (5, 25, 50, 75, 95):
    print(f"   p{q:<3} {np.percentile(err, q):+7.3f} ppm")
print(f"   |error| > 3 ppm : {np.mean(np.abs(err) > 3):.1%}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(np.clip(err, -8, 8), bins=80)
ax.axvline(np.median(err), color="k", ls="--",
           label=f"median {np.median(err):+.3f} ppm -- a real systematic bias")
ax.set_xlabel("precursor mass error (ppm)"); ax.set_ylabel("spectra"); ax.legend()
plt.tight_layout(); plt.show()

**The median is not zero.** The instrument reads systematically heavy — a calibration bias
worth knowing about, and worth being careful with: narrowing a mass window on the strength
of it is a good way to throw away real answers in the tails.

## Exact mass and isotopes

Most carbon atoms weigh exactly 12.000000 (that is ¹²C, the definition of the unit), but
about 1.1% weigh 13.00335 (¹³C).

- **Exact / monoisotopic mass** — computed using the *most common* isotope of every atom.
  This is what "neutral mass" means throughout.
- **Isotope pattern** — because some molecules contain a ¹³C, a small peak appears about
  1.00336 Da heavier. Its size indicates roughly how many carbons there are.

## Instrument families

**timsTOF**, **Orbitrap**, **QTOF**, **ion trap** — different physics, different accuracy,
different fragmentation. It matters because a reference spectrum from one machine does not
look quite like a query from another.

In [5]:
meta = pq.read_table(TRAIN, columns=["instrument_type", "inchikey14", "ingest_lib"]
                     ).to_pandas()
print("spectra per instrument family")
print()
for k, v in meta.instrument_type.value_counts().head(8).items():
    print(f"   {k:<26} {v:>10,}")
print(f"\ntotal {len(meta):,} spectra, {meta.inchikey14.nunique():,} distinct structures")
print("\nper source library -- note these ARE the major public MS/MS collections:")
print()
for k, v in meta.ingest_lib.value_counts().head(10).items():
    print(f"   {k:<22} {v:>10,}")
print("\nWorth absorbing: GNPS, MassBank, MoNA, RIKEN and MSDIAL are already in here.")
print("Bolting another public spectral library on top adds very little that is new.")

---
# Part 2 — Writing a molecule down

## SMILES

A molecule as a text string. `CCO` is ethanol. Rings are numbered (`c1ccccc1` is benzene,
lower case meaning aromatic), branches use brackets. **The same molecule has many valid
SMILES**, so you *canonicalise* — run an algorithm that always picks the same one.

## InChI and InChIKey

**InChI** is another text format, built for uniqueness rather than readability. The
**InChIKey** is a 27-character hash of it, in three blocks:

    RYYVLZVUVIJVGH-UHFFFAOYSA-N
    |____________| |_________| |
      skeleton      stereo      charge
       14 chars

**InChIKey14** is just the first block. Two molecules share it if they have the same atom
connectivity, *ignoring stereochemistry*. This competition scores on that first block, so
**you do not need to get stereochemistry right.**

In [6]:
for smi in ["CCO", "OCC", "C(C)O"]:
    m = Chem.MolFromSmiles(smi)
    print(f"  {smi:<8} -> canonical {Chem.MolToSmiles(m):<8} "
          f"InChIKey14 {Chem.MolToInchiKey(m)[:14]}")
print()
print("Three spellings, one molecule, one key.")

## Tautomers — and why the metric canonicalises them

A **tautomer** is the same molecule with a hydrogen sitting somewhere else, flipping a
double bond as it moves. The forms interconvert constantly in solution — arguably the same
substance — but they are drawn differently, so they get **different InChIKeys**.

That would make scoring arbitrary, so the metric runs a **tautomer canonicalisation**
first: both forms map to one representative before the key is taken. "The metric's key"
always means **tautomer-canonical InChIKey14**.

In [7]:
from rdkit.Chem.MolStandardize import rdMolStandardize
_TAUT = rdMolStandardize.TautomerEnumerator()

def structure_key(smiles):
    # exactly what the metric does: canonicalise tautomers, then take InChIKey block 1
    m = Chem.MolFromSmiles(smiles)
    return None if m is None else Chem.MolToInchiKey(_TAUT.Canonicalize(m)).split("-")[0]

for name, smi in (("keto form", "CC(=O)CC(=O)C"), ("enol form", "CC(O)=CC(=O)C")):
    m = Chem.MolFromSmiles(smi)
    print(f"  {name:<10} {smi:<16} raw {Chem.MolToInchiKey(m)[:14]}"
          f"   metric key {structure_key(smi)}")
print()
print("Raw keys differ; the tautomer-canonical keys agree -> scored as correct.")

## Fingerprints and Tanimoto

A **fingerprint** turns a molecule into a long vector of 0/1 bits: "does it contain this
substructure?" for thousands of substructures.

- **Morgan / ECFP** — each atom plus its neighbourhood out to radius *r*. The dominant type.
- **MACCS keys** — 167 hand-picked human-designed questions.
- **RDKit fingerprint** — path-based.

Two are compared with **Tanimoto similarity**, identical to the **Jaccard index**: bits
both have on, divided by bits either has on. 1.0 identical, 0.0 nothing shared. Loosely,
>0.7 is "clearly related", <0.3 "not obviously related".

In [8]:
from rdkit.Chem import rdFingerprintGenerator, DataStructs
mg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
mols = {"caffeine":    "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
        "theobromine": "Cn1cnc2c1c(=O)[nH]c(=O)n2C",   # caffeine minus one methyl
        "aspirin":     "CC(=O)Oc1ccccc1C(=O)O"}
fps = {k: mg.GetFingerprint(Chem.MolFromSmiles(v)) for k, v in mols.items()}
names = list(fps)
print(f"{'':<14}" + "".join(f"{n:>13}" for n in names))
for a in names:
    print(f"{a:<14}" + "".join(
        f"{DataStructs.TanimotoSimilarity(fps[a], fps[b]):>13.3f}" for b in names))
print()
print("caffeine/theobromine differ by one methyl -> high similarity.")

## Formula, isomers, RDBE

The **molecular formula** counts atoms: caffeine is C8H10N4O2. Molecules with the *same
formula* but different bonding are **isomers**, and they have **identical exact mass** — so
**no mass measurement can ever separate them.** Hold onto that; Part 6 shows it is the
whole ballgame.

**RDBE** (ring + double bond equivalents, a.k.a. degree of unsaturation) counts rings plus
double bonds, computed from the formula alone:

    RDBE = C - (H + halogens)/2 + N/2 + 1

Because it comes from the formula, **all isomers share it too.**

In [9]:
iso = {"glucose":  "OCC1OC(O)C(O)C(O)C1O",
       "fructose": "OCC1(O)OC(CO)C(O)C1O",
       "mannose":  "OCC1OC(O)C(O)C(O)C1O"}
print(f"{'molecule':<12}{'formula':<14}{'exact mass':>12}{'RDBE':>7}")
for n, smi in iso.items():
    m = Chem.MolFromSmiles(smi)
    c = {}
    for a in m.GetAtoms():
        c[a.GetSymbol()] = c.get(a.GetSymbol(), 0) + 1
    nH = sum(a.GetTotalNumHs() for a in m.GetAtoms())
    rdbe = c.get("C", 0) - nH / 2 + c.get("N", 0) / 2 + 1
    print(f"{n:<12}{rdMolDescriptors.CalcMolFormula(m):<14}"
          f"{ExactMolWt(m):>12.5f}{rdbe:>7.1f}")
print()
print("Same formula, same mass, same RDBE. Only fragmentation can separate them.")

---
# Part 3 — Finding candidates, and ranking them

## The candidate pool

You cannot invent molecules from nothing, so you keep a big list of known structures — the
**candidate pool**. For natural products that means a structure database such as
**COCONUT**, plus whatever structures appear in the training spectra.

What goes in the pool is one of the few genuinely load-bearing decisions: it fixes which
answers are reachable at all, and no amount of ranking recovers one that is missing.

## The mass window

Given a query's neutral mass, take every pool structure whose mass agrees within some ppm
tolerance. That set is the **candidate window**, and everything afterwards is *ordering* it.

## Ranking, and the channels

**Ranking** = sorting the window so the right answer is near the top. Most approaches score
each candidate on several independent kinds of evidence, called **channels**:

| channel | question it answers | works when |
|---|---|---|
| **library** | is this *exact* spectrum in our reference collection? | the molecule was measured before |
| **analog** | does a *similar* spectrum exist at a different mass? | a close relative was measured |
| **frag** | if I break this candidate in silico, do the pieces match the observed peaks? | always — no precedent needed |
| **neural** | a model predicts a fingerprint from the spectrum; how close is this candidate's? | always — no precedent needed |

Those become numerical **features**, fed to a **GBM** (gradient-boosted machine — an
ensemble of decision trees) producing a probability per candidate. Sort, take the top 25.

**Seed bagging** — train the GBM with several random seeds and average. Cheap variance
reduction, and worth a measurable amount here.

**Note the asymmetry.** *library* and *analog* need the molecule, or something like it, to
have been measured before. *frag* and *neural* do not. That single distinction decides
everything about the hard cases.

---
# Part 4 — The score: MRR@25

For each molecule you submit **25 ranked guesses**:

- find the position of the correct answer — that is the **rank** (1 = first guess)
- the **reciprocal rank** is `1/rank`
- not in your 25 at all → **0**
- **MRR@25** is the mean reciprocal rank over all molecules

Getting it first is worth **25×** getting it 25th. This shapes every design decision.

In [10]:
print(f"{'rank':>6}{'reciprocal rank':>18}{'value vs rank 1':>18}")
for r in [1, 2, 3, 4, 5, 10, 13, 20, 23, 25]:
    print(f"{r:>6}{1/r:>18.4f}{1/r:>17.1%}")
tail = sum(1 / k for k in range(13, 26)) / 13
print()
print(f"ranks 13-25 average {tail:.4f} each -- about 5% of a rank-1 hit.")

## "The tail"

**The tail** is the bottom of your list, roughly ranks 13–25. Because a rank-23 hit is
worth 0.043 against a rank-1 hit's 1.0, the tail is *nearly free to give away*.

Measured on my 250-molecule natural-product holdout — truncating the list and rescoring:

| retrieval slots kept | MRR@25 | cost vs keeping 25 |
|---|---|---|
| 25 | 0.72856 | +0.00000 |
| 22 | 0.72856 | +0.00000 |
| 16 | 0.72838 | −0.00018 |
| 13 | 0.72809 | −0.00047 |
| 11 | 0.72742 | −0.00113 |
| 10 | 0.72706 | −0.00150 |
|  5 | 0.71987 | −0.00869 |
|  1 | 0.60000 | −0.12856 |

**Throwing away slots 13–25 entirely costs about 0.0005 of MRR.** The truth essentially
never lands there — if the ranker was going to find it, it found it much higher.

That cuts both ways, and the second edge is the one people miss. **Free slots are free
because they are worthless.** Anything you put in the tail can only earn about 5% of a
rank-1 hit, so the ceiling on any tail-filling trick is small however clever it is.

---
# Part 5 — De novo generation, and BRICS

## Retrieval vs de novo

- **Retrieval** — pick from a list of known structures.
- **De novo** — propose a structure nobody gave you.

If the answer is in no database you hold, retrieval **cannot** produce it, at any rank,
ever. De novo is the only route.

## BRICS — how to invent a molecule you were never given

**BRICS** (*Breaking of Retrosynthetically Interesting Chemical Substructures*) is a set of
16 rules for cutting a molecule at bonds a chemist would actually make or break — an amide
bond, an ether linkage, a ring–substituent bond. The cuts are chosen so the pieces are
sensible building blocks.

Each cut end is **labelled** with a number recording what kind of chemistry it was, and the
rules say which labels may be rejoined. That bookkeeping is what stops recombination
producing nonsense.

The generation loop:

1. **Cut** a known molecule (a *donor*) into labelled pieces, each with a mass.
2. **Keep** one piece. You now need a partner weighing `target_mass − piece_mass` with a
   compatible label.
3. **Look it up** in a **fragment index** — every fragment from tens of thousands of
   molecules, bucketed by (label, mass). The index's source chemistry matters: an index
   built from drug-like molecules cannot supply natural-product fragments.
4. **Glue** them. The product lands at the target mass *by construction*.

The result may exist in no database at all — which is the only way to reach the hard cases.

In [11]:
from rdkit.Chem import BRICS
donor = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")      # paracetamol, a simple donor
print("donor:", Chem.MolToSmiles(donor))
print()
print("BRICS bonds RDKit finds (atom pair -> label pair):")
for (a, b), (la, lb) in BRICS.FindBRICSBonds(donor):
    print(f"   atoms {a:>2},{b:<2}  labels {la} - {lb}")

frags = sorted(BRICS.BRICSDecompose(donor))
print()
print(f"fragments ({len(frags)}); [n*] marks a labelled cut end:")
for f in frags:
    m = Chem.MolFromSmiles(f, sanitize=False)
    m.UpdatePropertyCache(strict=False)    # BRICS fragments are not fully sanitizable
    print(f"   {f:<28} mass {ExactMolWt(m):8.4f}")

Now recombine — hand BRICS two fragments and it returns every legal join:

In [12]:
pieces = [Chem.MolFromSmiles(f, sanitize=False) for f in frags[:2]]
print("gluing:", [Chem.MolToSmiles(p) for p in pieces])
print()
seen = set()
for built in BRICS.BRICSBuild(pieces, maxDepth=0, scrambleReagents=False):
    Chem.SanitizeMol(built)
    smi = Chem.MolToSmiles(built)
    if smi in seen:
        continue
    seen.add(smi)
    print(f"   {smi:<34} mass {ExactMolWt(built):9.4f}")
    if len(seen) >= 8:
        break
print()
print(f"{len(seen)} distinct products from two fragments. Swap the partner for a")
print("DIFFERENT fragment of the same mass and label and you get a molecule at the")
print("same precursor mass that nobody has ever recorded.")

**The catch, and it is the whole difficulty.** This produces *hundreds* of candidates per
molecule at the right mass, all needing to be ranked against the retrieved ones. Three
things I measured the hard way:

- A generated structure has **no library spectrum by definition**, so the strongest
  evidence channel scores it zero, along with anything derived from it.
- A BRICS product is assembled **out of the very donors it is then compared against**, so a
  fingerprint-similarity score is partly measuring your own construction. I measured the
  inflation at **+0.516 standard deviations** over an honestly-retrieved candidate. It wins
  on the scale, not on the evidence — check your two populations are even comparable before
  merging them.
- On the molecules where retrieval is strong, the generator largely **re-derives retrieval's
  own top answer** — in my measurement, 88.8% of the time, at median rank 2. So a
  generator's standalone score is not simply additive with a retrieval score; a good part of
  it is the same answers counted twice.

---
# Part 6 — The competition's own jargon

## Class 1, class 2, class 3

The single most important idea here, and it describes **what you hold**, not a property of
the molecule:

| class | meaning | can you possibly get it right? |
|---|---|---|
| **class 1** | the molecule's *own spectra* are in your reference library | yes — nearly always |
| **class 2** | its *structure* is in your candidate pool, but no spectrum of it exists | yes — it is in the window, you just have to rank it |
| **class 3** | **neither** — the structure is in no database you hold | **no.** You emit only pool structures, so you score **exactly 0** |

Two consequences drive everything:

1. **Class 3 is reachable only by generating a structure you were never given.**
2. **Enlarge the pool and molecules move from class 3 to class 2.** The classes are not
   fixed properties of the test set; they shift when your data changes. So any "class-3
   share" you compute is a statement about *your* pool on *that* day.

## Why class 3 cannot be measured locally — and this is the trap

To test a class-3 method you need natural products *absent* from the natural products
database. Count them:

    natural-product structures with >=2 spectra :    220
      present in the COCONUT gallery            :    219
      ABSENT -> usable as class-3 test cases     :      1

**COCONUT *is* the natural-products database**, so essentially every natural product you
have labels for is already in it. A natural-product class-3 holdout cannot be built — by
anyone.

Every class-3 number therefore gets measured on *drug-like* molecules as a stand-in. Mine
have been wrong **5 times out of 5** against the real leaderboard. If you take one thing
from this notebook, take that: **a class-3 result measured on drug-like chemistry does not
transfer, and there is no local fix.**

## Holdout, regimes, leakage

A **holdout** is data you hide from yourself. Mine takes 250 natural-product molecules and
**deletes every spectrum of them from every reference library** — otherwise library search
just finds the answer. That is **leakage**: test data reaching your training data, making
results look far better than they are. In this competition it is unusually easy to
self-inflict, because the reference library is enormous and the same molecule appears in
several source collections.

Within such a holdout:

- **regime A** — a spectrum of the exact structure is still available (simulates class 1)
- **regime B** — every spectrum deleted (simulates class 2)

## Named models you will see

| name | what it is |
|---|---|
| **MIST** | a published spectrum → fingerprint model |
| **ICEBERG** | a published model that *predicts* how a molecule fragments |
| **sylva** | another spectrum → fingerprint model used in this competition |
| **MetFrag** | the classic in-silico fragmenter; most `frag` channels are a small version |

A caution on MIST specifically: it needs the true **molecular formula** as an input.
`test.parquet` has no formula column, so at inference you must predict it (msbuddy gets
~56% top-1), and a wrong formula destroys the fingerprint completely rather than degrading
it gracefully.

## Board vocabulary

- **the board** / **LB** — the public leaderboard.
- **CV** — cross-validation, your own offline estimate. **Not the same thing**, and in this
  competition they disagree badly.
- **the instrument** — my word for whatever offline measurement you use to predict the
  board.

---
# Part 7 — The two things that cost me the most time

## 1. Almost everything beating the truth is an isomer

Measured over every candidate my ranker put **above** the correct answer:

    candidates ranked ABOVE the truth :  286
      with the SAME molecular formula :  277   (96.9%)

Same formula ⇒ same exact mass, same RDBE, same nitrogen parity, same H/C ratio. **No
feature computed from the formula can separate them, even in principle.** I measured five
such features and every one came back at chance:

    RDBE 0.5016   nitrogen-rule 0.4958   H/C 0.5119   O/C 0.5123   N/C 0.5120
    (AUC; 0.50 is a coin flip)

That is not weak signal — it is *no possible signal*. Only **fragmentation** distinguishes
isomers, so that is where the remaining headroom lives.

## 2. Offline measurements here do not predict the board

My own record, honestly:

| offline prediction | what the board did |
|---|---|
| a fragmentation channel, +0.1483 over the incumbent | **−0.006** |
| a mass-window correction, −26% candidates | **−0.003** |
| a drug-like holdout, five separate predictions | **0 for 5** |
| an interleaving ceiling of +0.0095 | **−0.035** |
| a fragment-index source change, +15.5 points of reach | **+0.004** |

**One success in six.** Every offline effect I can measure is smaller than the demonstrated
error of the thing measuring it.

Two specific traps inside that record, both worth stealing:

- **Never argue for swapping a channel from the channel's intrinsic metric.** My own
  fingerprint model beats a public one by 26.5% on fingerprint accuracy and *loses* by
  0.1353 of MRR as a ranking channel, on identical candidate sets. The intrinsic metric is
  a property of the model; ranking is a property of the model *and* the decoys it must beat.
- **Check your validation population matches the test population** — not just the chemistry,
  but the *window sizes*. My holdout averages 82.8 candidates per query; the real test
  averages 131.6, and accuracy falls hard as the window grows. A gain measured on small
  windows can vanish entirely on large ones.

---
# Quick reference

| term | one line |
|---|---|
| **m/z** | mass-to-charge ratio, what the instrument measures |
| **adduct** | the charged thing the molecule became, e.g. `[M+H]+` |
| **neutral mass** | the molecule's own mass, recovered from m/z and the adduct |
| **precursor** | the intact ion selected for fragmentation |
| **fragment** | a piece produced by smashing it |
| **collision energy** | how hard it was smashed, in eV |
| **base peak** | most intense peak; intensities are scaled to it |
| **ppm** | relative mass error, parts per million |
| **monoisotopic / exact mass** | mass using the commonest isotope of each atom |
| **isotope pattern** | the small heavier peaks from ¹³C etc.; hints at carbon count |
| **SMILES** | a molecule written as a text string |
| **InChIKey14** | first block of a structure hash; ignores stereochemistry |
| **tautomer** | same molecule, hydrogen elsewhere; canonicalised by the metric |
| **fingerprint** | bit vector of substructure presence (Morgan/ECFP, MACCS) |
| **Tanimoto / Jaccard** | fingerprint similarity, 0 to 1 |
| **isomer** | same formula, different structure — *identical mass* |
| **RDBE** | rings + double bonds, from the formula alone |
| **candidate pool** | every structure you are willing to guess |
| **mass window** | pool entries whose mass matches the query |
| **channel** | one source of evidence: library, analog, frag, neural |
| **GBM** | gradient-boosted trees; ranks candidates from the features |
| **seed bagging** | average several GBMs trained with different seeds |
| **MRR@25** | mean of 1/rank; 0 if not in the top 25 |
| **the tail** | ranks ~13–25, each worth ~5% of a rank-1 hit |
| **class 1 / 2 / 3** | spectra in library / structure in pool / neither |
| **holdout** | data hidden from yourself to test honestly |
| **regime A / B** | holdout molecules with / without their own spectra |
| **leakage** | test data contaminating training, inflating results |
| **retrieval** | pick from known structures |
| **de novo** | propose a structure nobody gave you |
| **BRICS** | cut molecules at sensible bonds, recombine the pieces |
| **fragment index** | fragments bucketed by (label, mass) for fast partner lookup |
| **donor** | the known molecule a generated candidate was built from |
| **the board** | the public leaderboard |
| **the instrument** | whatever offline measure you use to predict the board |